# Spam Email / SMS Classifier

NLP classification on the [UCI SMS Spam Collection](https://archive.ics.uci.edu/ml/datasets/SMS+Spam+Collection)
(Almeida et al., 2011): 5,574 real SMS messages labeled **ham** / **spam**.

**Task:** classify a message as spam (1) or ham (0).
**Approach:** TF-IDF features → three classic classifiers (Naive Bayes, Logistic Regression, Linear SVM),
compared on accuracy, precision, recall, F1 and ROC-AUC — with 5-fold CV for stability.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import clean_text, load_data
from src.models import build_vectorizer

sns.set_theme(style="whitegrid")

## 1. Load & inspect

In [2]:
df = load_data("data")
df["clean"] = df["message"].map(clean_text)
print(f"Messages: {len(df)} | spam rate: {df['label'].mean():.1%}")
df.head(8)

Messages: 5574 | spam rate: 13.4%


,label,message,clean
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis great wo...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry NUM wkly comp win fa cup final tkts...
3,0,U dun say so early hor... U c already then say...,dun say early hor already say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah don think goes usf lives around though
5,1,FreeMsg Hey there darling it's been 3 week's n...,freemsg hey darling NUM week now word back lik...
6,0,Even my brother is not like to speak with me. ...,even brother like speak treat like aids patent
7,0,As per your request 'Melle Melle (Oru Minnamin...,per request melle melle oru minnaminunginte nu...


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
df["len"] = df["message"].str.len()
sns.histplot(df[df.label==0]["len"], bins=40, color="#0f6b8c", ax=axes[0], label="ham")
sns.histplot(df[df.label==1]["len"], bins=40, color="#e07a5f", ax=axes[0], label="spam")
axes[0].set_title("Message length distribution"); axes[0].legend()
sns.countplot(data=df, x="label", hue="label", palette=["#0f6b8c", "#e07a5f"], ax=axes[1], legend=False)
axes[1].set_xticklabels(["Ham", "Spam"])
axes[1].set_title("Class balance")
plt.tight_layout(); plt.show()
print(f"Avg length: ham={df[df.label==0]['len'].mean():.0f} chars, spam={df[df.label==1]['len'].mean():.0f} chars")

Avg length: ham=71 chars, spam=139 chars


/var/folders/gn/6xy38n2d1qsf3l0fysxxxp100000gn/T/ipykernel_73310/1246120022.py:7: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  axes[1].set_xticklabels(["Ham", "Spam"])
/var/folders/gn/6xy38n2d1qsf3l0fysxxxp100000gn/T/ipykernel_73310/1246120022.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Insight:** spam messages are typically *longer* (more promotional detail) — length is a weak signal;
the real signal is vocabulary.

## 2. Train & compare classifiers

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split, cross_val_score

X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.2, random_state=42, stratify=df["label"])

models = {
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "Logistic Regression": LogisticRegression(max_iter=2000, C=1.0, random_state=42),
    "Linear SVM (SGD)": SGDClassifier(loss="modified_huber", random_state=42, max_iter=2000),
}
results = []
for name, model in models.items():
    pipe = Pipeline([("tfidf", build_vectorizer()), ("clf", model)]).fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = (pipe.decision_function(X_test) if hasattr(pipe.named_steps["clf"], "decision_function")
              else pipe.predict_proba(X_test)[:, 1])
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
    })

pd.DataFrame(results).round(4)

,model,accuracy,precision,recall,f1,roc_auc
0,Naive Bayes,0.9776,0.9844,0.8456,0.9097,0.9905
1,Logistic Regression,0.9785,0.9921,0.8456,0.9130,0.9909
2,Linear SVM (SGD),0.9848,0.9648,0.9195,0.9416,0.9921


## 3. 5-fold cross-validation (F1 stability)

In [5]:
cv_rows = []
for name, model in models.items():
    pipe = Pipeline([("tfidf", build_vectorizer()), ("clf", model)])
    cv = cross_val_score(pipe, df["clean"], df["label"], cv=5, scoring="f1")
    cv_rows.append({"model": name, "cv_f1_mean": cv.mean(), "cv_f1_std": cv.std()})
pd.DataFrame(cv_rows).round(4)

,model,cv_f1_mean,cv_f1_std
0,Naive Bayes,0.9217,0.0116
1,Logistic Regression,0.8945,0.0252
2,Linear SVM (SGD),0.9419,0.0115


## 4. Best model: confusion matrix + most indicative words

In [6]:
from sklearn.metrics import ConfusionMatrixDisplay

best = Pipeline([
    ("tfidf", build_vectorizer()),
    ("clf", SGDClassifier(loss="modified_huber", random_state=42, max_iter=2000)),
]).fit(X_train, y_train)
y_pred = best.predict(X_test)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues",
                                        display_labels=["Ham", "Spam"], ax=ax)
plt.title("Linear SVM — Confusion Matrix")
plt.show()

/var/folders/gn/6xy38n2d1qsf3l0fysxxxp100000gn/T/ipykernel_73310/2771499343.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
coefs = best.named_steps["clf"].coef_[0]
vocab = np.array(best.named_steps["tfidf"].get_feature_names_out())
order = np.argsort(coefs)
all_words, all_coefs = vocab[order], coefs[order]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].barh(all_words[:15][::-1], all_coefs[:15][::-1], color="#0f6b8c")
axes[0].set_title("Top ham-indicative words")
axes[1].barh(all_words[-15:][::-1], all_coefs[-15:][::-1], color="#e07a5f")
axes[1].set_title("Top spam-indicative words")
plt.tight_layout(); plt.show()

/var/folders/gn/6xy38n2d1qsf3l0fysxxxp100000gn/T/ipykernel_73310/637728457.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Try it on your own message

In [8]:
def classify(message: str):
    p = best.predict_proba([clean_text(message)])[0, 1]
    return "SPAM" if p > 0.5 else "HAM", p

for msg in ["Congratulations! You won a free iPhone, text CLAIM now!",
            "Are we still meeting for lunch tomorrow at 12?"]:
    verdict, prob = classify(msg)
    print(f"{verdict} ({prob:.1%})  <-  {msg}")

SPAM (100.0%)  <-  Congratulations! You won a free iPhone, text CLAIM now!
HAM (0.0%)  <-  Are we still meeting for lunch tomorrow at 12?


## 6. Summary

| Model | Accuracy | Precision | Recall | F1 | AUC | 5-fold CV F1 |
|---|---|---|---|---|---|---|
| Naive Bayes | 97.8% | 98.4% | 84.6% | 0.910 | 0.991 | 0.92 ± 0.01 |
| Logistic Regression | 97.9% | 99.2% | 84.6% | 0.913 | 0.991 | 0.89 ± 0.03 |
| **Linear SVM (SGD)** | **98.5%** | **96.5%** | **92.0%** | **0.942** | **0.992** | **0.94 ± 0.01** |

**Winner: Linear SVM.** It catches 92% of all spam (vs 84.6% for the others) while keeping
96%+ of real messages safe — the critical trade-off in spam filtering is **recall on spam**,
which the margin-based SVM handles best.

> Run the live demo: `streamlit run app/app.py`